# Cleaning Rules — djelfa.info

Analyzes the representative djelfa.info sample (reuses the cached sample from `01_sample_and_explore.ipynb` — 1,000 randomly-reservoir-sampled forum posts, already representative) to find recurring noise patterns and derive a concrete, documented set of cleaning rules. Output of this notebook is the **rules themselves** (final section), not a cleaning implementation.

## 1. Setup

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

ROOT = Path.cwd().parent  # Notebooks/ -> Darija/
SAMPLE_PATH = ROOT / "Data" / "sample_djelfa_info.jsonl"

if not SAMPLE_PATH.exists():
    raise FileNotFoundError(
        f"{SAMPLE_PATH} not found — run 01_sample_and_explore.ipynb first to build the cached sample."
    )

with SAMPLE_PATH.open("r", encoding="utf-8") as f:
    df = pd.DataFrame(json.loads(line) for line in f if line.strip())

print(f"loaded {len(df)} sampled djelfa.info posts")


def mask_for(pattern):
    """bool Series for whether `pattern` matches — avoids pandas' str.contains
    warning about patterns with capture groups, and avoids greedy-backtracking
    traps that a single combined regex can fall into (confirmed one of these
    in the YouTube notebook — see that notebook's mention-glue check)."""
    return df["text"].apply(lambda t: bool(pattern.search(t)))


def report_mask(name, mask, n_examples=8):
    hits = df[mask]
    pct = len(hits) / len(df) * 100
    print(f"=== {name}: {len(hits)}/{len(df)} ({pct:.1f}%) ===")
    for text in hits["text"].sample(min(n_examples, len(hits)), random_state=0):
        preview = text[:200].replace("\n", " ")
        print(f"- {preview!r}")
    print()
    return mask


def report(name, pattern, n_examples=8):
    return report_mask(name, mask_for(pattern), n_examples=n_examples)

loaded 1000 sampled djelfa.info posts


## 2. Quote-block boilerplate

vBulletin's reply-quoting wrapper: `اقتباس:\nالمشاركة الأصلية كتبت بواسطة <username>\n<quoted text>`. Already spotted repeatedly in notebook 01's output. The wrapper phrase itself is pure boilerplate; the quoted text underneath is a repost of someone else's earlier post (dedup should already catch it as near-duplicate elsewhere, but the wrapper text pollutes every occurrence regardless).

In [2]:
QUOTE_WRAPPER_RE = re.compile(r"اقتباس:\s*\nالمشاركة الأصلية كتبت بواسطة\s+\S+")

mask = report("Quote-block wrapper (اقتباس: المشاركة الأصلية كتبت بواسطة ...)", QUOTE_WRAPPER_RE)

# How much of the *document* is typically the quoted wrapper text vs. the
# poster's own reply after it? (rough: split on the wrapper, look at what's left)
lengths_after = []
for text in df.loc[mask, "text"]:
    parts = QUOTE_WRAPPER_RE.split(text, maxsplit=1)
    if len(parts) > 1:
        lengths_after.append(len(parts[0]) + len(parts[-1]))
print("char count outside the quote wrapper (for docs containing it):")
print(pd.Series(lengths_after).describe())

=== Quote-block wrapper (اقتباس: المشاركة الأصلية كتبت بواسطة ...): 239/1000 (23.9%) ===
- 'اقتباس: المشاركة الأصلية كتبت بواسطة abousoumia وعندما تقراون اسم عضويتي تجدونه باللاتينية ابو سمية , لكن لتعلمو انني لست ابا بعد انما ابو سمية هو والدي , وسمية هي اختي الصغرى اما انا فاسمي انس تلميذ '
- 'اقتباس: المشاركة الأصلية كتبت بواسطة mofida13 اللسلام عليكم ورحمة الله وبركاته احم احم قررت ان اعطيك استراحة لاني رحمتك لما رايت انظمام المحققة (ام عاكف)الى جانبي __يافرحتي__ ولما رايت ايضا الكم الذي '
- 'اقتباس: المشاركة الأصلية كتبت بواسطة chercheur eco بنيتي إتخذي من الصمت نقطة قوّة بدل نقطة ضعف وماذا بعد إن كنت أنانية فحبّ النفس دون إفراط ودون تفريط واجب علينا فإن لم تحبّي نفسك لن تحفظي كرامتها وإن'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة rowidaPhil الله يعيد رمضان علينا بكل خير يارب اااااااااااااامين شكرا'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة نور لاتراه السلام عليكم مرحبا بالاخت ميم والشكر يتجدد لجميلتنا الجميلة على حسن انتقائها استضافة طيبة ارجوها لكم وَ عَلَيْك السّلام أُخْتي نُور شُك

## 3. BBCode leftover tags

vBulletin markup that didn't get rendered to plain text — `[COLOR="Sienna"]`, `[size=2]`, `[font=Arial]`, etc. Already spotted in notebook 01's output.

In [3]:
BBCODE_RE = re.compile(r"\[/?[a-zA-Z]+(?:=[^\]]*)?\]")

mask = report("BBCode tags", BBCODE_RE)

all_tags = [m.group(0) for t in df["text"].dropna() for m in BBCODE_RE.finditer(t)]
print("most common tags:", Counter(all_tags).most_common(15))

=== BBCode tags: 11/1000 (1.1%) ===
- '[COLOR="Sienna"] [size=2][font=Arial] مرحبا بك أخت نينا و بكل سكان البليدة  في    منتدانا لا تحزني  أنا  معاك تم حذف الايميل  من طرف الادارة اللهم صل على محمد وعلى آله وصحبه أجمعين'
- 'هل فعلا المرأة برأيك مخلوق لن يفهم يوما وما تعليقك على هذه الصورة [IMG] [/IMG]'
- '[QUOTE=أمير جزائري حر;3998090993] تحية طيبة أخت تو .. ليس عليك أن تحدّي من [الغيرة] / وإن فعلتِ فقد تكونين بصدد ارتكاب جناية بحق ذلك [الإنسان] .. كل ما عليك هو [توجيهها] = la canaliser .. توجيهها [بذك'
- '1 [/B][/SIZE] بني ادم و ابليس لفت انتباه للقارئ الكريم اذكر سؤالك او رئيك او انتقادك او معلومتك التي يمكن ان نستفيد منها في التعليقات اسفله فيكون بذلك تفاعل بين القراء الكرام  و منفعة لنا جميعا وربما '
- 'سابعا: نتائج تقديس الوطنية أثمرت الدعوة إلى الوطنية ثمارا خبيثة وبرزت العصبية البغيضة وانتزعت الرحمة بين الناس وحل محلها الفخر والخيلاء والكبرياء حيث تعصب كل شعب لوطنه واحتقر ما عداه في صور مخزية مفرق'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة أخت الرجال السلام عليكم ورحمة الله تعالى 

## 4. URLs

Note: same as YouTube — already slated for anonymization (placeholder, don't delete) per the project's plan. Sizing it here.

In [4]:
URL_RE = re.compile(r"(?:https?://\S+|www\.\S+)", re.IGNORECASE)

report("URLs", URL_RE)

=== URLs: 16/1000 (1.6%) ===
- 'نقل في القمة جزاك الله كل خير أخي بالمناسبة لديك بعض الاسئلة رجاءا ادخل هنا لانني لا استطيع مراسلتك https://www.djelfa.info/vb/showthrea...2239348&page=2'
- 'مر من منا أخوكم ب. علي https://www.djelfa.info/vb/member.php?u=142497'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة أبو معاذ محمد رضا السلام عليكم كنت في السابق أحمّل من موقع فور شارد بكل سهولة والآن لا أستطيع ما العمل؟ أخي https://4shared.com/ اصبح يطلب فتح حساب لتحميل منه يعني'
- 'لكل من يعرف غاليتي حبيبتي زينب ولكل من لا يعرفها www.djelfa.info/vb/member.php?u=343212 زينب في ذمّة الله لقد توفيت اليوم حوالي الساعة السادسة والنصف صباحا يوم الجمعة 20 جويلية 2012 الموافق لــ 01 رمض'
- 'يانابغتنا شكراااا للاجابة واقترح عليك المشاركة في مسابقة مني الفكرة ومنكم الموضع تجدينها في قسم الجلفة للمواضيع العامة وهذه المسابقة عبارة عن موضوع مثبت .......هذا هو الرابط ..................شاركي ht'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة samsungsos شوفي اسعار condor اهنا https://3galgerien.com/3g11539.html شكرا على المسا

0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

## 5. HTML entities

Leftover unescaped entities (`&amp;`, `&quot;`, `&nbsp;`) from old forum content that may not have been fully decoded.

In [5]:
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z]+;|&#\d+;")

mask = report("HTML entities", HTML_ENTITY_RE)
all_entities = [m.group(0) for t in df["text"].dropna() for m in HTML_ENTITY_RE.finditer(t)]
print("most common entities:", Counter(all_entities).most_common(10))

=== HTML entities: 0/1000 (0.0%) ===

most common entities: []


## 6. Repeated/spam emoji runs

In [6]:
EMOJI_CLASS = (
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U0001F1E6-\U0001F1FF"
    "☀-➿"
)
EMOJI_CHAR_RE = re.compile(f"[{EMOJI_CLASS}]")
EMOJI_RUN_RE = re.compile(f"(?:[{EMOJI_CLASS}][️‍]?){{3,}}")

report("3+ consecutive emoji", EMOJI_RUN_RE)

emoji_counts = df["text"].apply(lambda t: len(EMOJI_CHAR_RE.findall(t)))
print("emoji-count distribution (per post):")
print(emoji_counts.describe())

=== 3+ consecutive emoji: 8/1000 (0.8%) ===
- 'اقتباس: المشاركة الأصلية كتبت بواسطة هديل السلام happy birthday my sweetest friend in the universe* كل عام وانت بالف الف خير عقبال 100سنة* احلى مها بالدنيا 😘😘😘😘😘😘 شكرا هدييل على مرورك🤩 البقية بحياتك و'
- 'كانت هنا عضوة تعشق منتدى الجلفة وتملك من العمر 11سنة اسمها ♥ياسمين عاشقة العلم♥ ادعو لها بالتوفيق ف دراستها والمرتبة الأولى في قسمها ♥♥♥تحياتي♥♥♥'
- 'السلام عليكم ورحمة الله وبركاته الأخ black dark knight أعتذر عن الخطأ الفادح بل أنت رجل ونص و25☺ كما أنك شهم وإلا لخصمت العلامة كلها☺ الأخت الكريمة ahlem chrd سؤالك غير موجه لي☺ عدّليه لأعرف كيف أرد☺ '
- 'كورية مستذئبة أرحب بالأخت "الوردة البيضاء " مع انني انضممت حديثا ربي يحفظك حبيبتي الكورية سعيدة انا بترحيبك الطيب * لدي بعض الاسئلة الخفيفة أهلا بك وبأسئلتك الخفيفة😊 - ماهو هدفك في الحياة؟ سبق وأجبت ع'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة متفائلة في زمن الياس اسمي كما اسمك نفولة ههه عاشت الاسامي حوحو شكر روحو 😁😉 انا كنت نحب كوكب زمرد كثير كيما سالي والحديقة السرية ريمي   لحن الحياة .... 

## 7. Elongated character runs

Same nuance as YouTube: expressive stretching (laughter, emphasis), not noise — collapse, don't strip.

In [7]:
ELONGATION_RE = re.compile(r"(\w)\1{2,}", re.UNICODE)

report("Elongated characters (3+ repeats)", ELONGATION_RE)

all_runs = [m.group(0) for t in df["text"].dropna() for m in ELONGATION_RE.finditer(t)]
print("most common elongated runs:", Counter(all_runs).most_common(15))

=== Elongated characters (3+ repeats): 168/1000 (16.8%) ===
- 'قال العلامة المحدث الألباني - رحمه الله رحمة واسعة وأسكنه الله فسيح جناته وجزاه الله عن الإسلام والمسلمين خيرا - في ( السلسة الصحيحة ) تحت جديث : أحاديث في تحريك الإصبع في التشهد،والرد على من أنكره 31'
- 'الشيماء بنت الحارث السعدية ، امرأة بدوية من بني سعد . - وهي ابنة حليمة السعدية التي كانت من بين مراضع بني سعد حين انطلقن إلى مكة يلتمسن الأطفال لإرضاعهم ، فلم يطل مكثها بمكة حتى عادت تحمل معها طفلاً ،'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة السيدة فيروز 4444444444444444444444'
- 'و عليكم سلام الله و رحمته و بركاته أهلا و سهلا  بي بينكم في رحاب مجهر الخيمة ما ظننتُ يومًا أني سأسقط ، الاّ أني سقطت [ و يا لها من وقعة ههه \\ الله المستعان ] أشكر زميلي كيان على  الاستضافة الجميلة   '
- 'اقتباس: المشاركة الأصلية كتبت بواسطة ~ غيمة ~ استمتعت حقا  بالاجابات شكرا ^^ انتظرك في اي وقت هههه الحمد لله انك استمتعتي العفو'
- 'اقتباس: المشاركة الأصلية كتبت بواسطة حمزة اليوم  أنت مراقبة و ربما في الغد القريب تصبحين على رأس هرم الإدارة و ل

## 8. Excessive punctuation runs

In [8]:
PUNCT_RUN_RE = re.compile(r"([!?؟.,])\1{2,}")

report("Excessive punctuation (!!!,؟؟؟,...)", PUNCT_RUN_RE)

=== Excessive punctuation (!!!,؟؟؟,...): 240/1000 (24.0%) ===
- 'اقتباس: المشاركة الأصلية كتبت بواسطة مِيمْ ادام الله صداقتكما ^_^ شكرا حبيبتي بارك الله فيك ...'
- 'تدريبات التركيز البصري نظلم غرفة و نجعل اتجاه الطفل ناحية حائط خالي من الصور الرسومات، و يكون المدرب خلف الطفل و معه كشاف نور و عمل التالي: توجيه نور الكشاف ناحية الحائط و تحريكه علي الحائط ببطء ( فوق'
- 'نقل في القمة جزاك الله كل خير أخي بالمناسبة لديك بعض الاسئلة رجاءا ادخل هنا لانني لا استطيع مراسلتك https://www.djelfa.info/vb/showthrea...2239348&page=2'
- 'كانوا هنا ........ ثم غابوا ...... توحشتكم برشا برشا'
- 'أترين كيف يختلط الفرح بالحزن فتحتارين على أي جانب تتكئين، هذا هو حالي الآن، وقد كنت أظن أننا سنمضي في الطريق ذاته ونتشارك ما يستلزم التشارك، دون أن يفقد أحدنا الآخر..  لكنني اليوم وعلى عتبة فقدك..  لا'
- 'اللهم لا تخرجنا من يومنا هذا إلا وأنت راض عنا... اللهم لا تخرجنا من يومنا هذا إلا بذنب مغفور... اللهم لا تخرجنا من يومنا هذا إلا بعمل مقبول... اللهم إبعدنا عن القلوب القاسية وإزرع فينا قلبا حنونا... ا'
- 'مرحبا

0       True
1       True
2      False
3      False
4      False
       ...  
995     True
996    False
997     True
998    False
999    False
Name: text, Length: 1000, dtype: bool

## 9. Auto-generated forum widget leakage

Spotted in notebook 01's output — an "most users online" style stat block (`أكبر تواجد بالمنتدى كان: 25,091 بتاريخ ...` followed by a long list of usernames) that looks like a sidebar/footer widget, not real post content. Checking whether this was a one-off or a real recurring scrape artifact.

In [9]:
WIDGET_LEAK_RE = re.compile(r"أكبر تواجد بالمنتدى كان")

report("Forum-widget leakage (أكبر تواجد بالمنتدى)", WIDGET_LEAK_RE, n_examples=3)

=== Forum-widget leakage (أكبر تواجد بالمنتدى): 4/1000 (0.4%) ===
- 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 12:15 \u200fزهرة المسيلة, \u200fabouyounes, \u200ftorab12, \u200f*عبدالرحمن*, \u200fkhalide, \u200fnono3, \u200fasmt, \u200fabdou-lad, \u200fأبو أشْرف, \u200flaouiyacine, \u200fbissa40, \u200fchanfawa, \u200f'
- 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 11:15 \u200fزهرة المسيلة, \u200fالأخضر48, \u200fMOHAMMED.AMIN21, \u200fنسيم الشوق, \u200fmisa39, \u200fbahi65b+, \u200fnadir2006, \u200fالباشق, \u200fnihal159, \u200famine1962, \u200fبن مير سليمان, '
- 'مشاهدة المتواجدون الآن أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 11:15 \u200fزهرة المسيلة, \u200fdz-yac, \u200fأبو سارة عبد اللطيف, \u200fsaid wail, \u200fأم إسلام, \u200fAMIRA76, \u200fأستاذ شاوي, \u200fArkham, \u200fH@liM, \u200fأبوع'



0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

## 10. Near-empty after stripping emoji/punctuation

Sizing the min-length filter already planned in the project docs, same as the YouTube notebook.

In [10]:
NON_WORD_RE = re.compile(r"[^\w؀-ۿ]", re.UNICODE)


def residual_letters(text):
    no_emoji = EMOJI_CHAR_RE.sub("", text)
    no_punct = NON_WORD_RE.sub("", no_emoji)
    return len(no_punct.strip())


residual = df["text"].apply(residual_letters)
mask = residual <= 1
report_mask("Near-empty after stripping emoji/punctuation", mask)
print(f"residual-length distribution:\n{residual.describe()}")

=== Near-empty after stripping emoji/punctuation: 0/1000 (0.0%) ===

residual-length distribution:
count     1000.000000
mean       483.736000
std       1300.528474
min          3.000000
25%         54.000000
50%        127.000000
75%        353.250000
max      21010.000000
Name: text, dtype: float64


## 11. Derived cleaning rules (djelfa.info)

Based on prevalence + examples above, on the 1,000-post sample:

| # | Pattern | Prevalence | Action | Rationale |
|---|---|---|---|---|
| 1 | Quote wrapper (`اقتباس:\nالمشاركة الأصلية كتبت بواسطة <user>`) | **23.9%** | Strip the wrapper phrase (keep whatever follows) | By far the most common noise pattern in this corpus. After stripping, real remaining content is substantial (median 341 chars, min 14) — so this is a targeted strip, not a "drop the whole doc" situation. The quoted text underneath is someone else's earlier post; dedup should catch it elsewhere as near-duplicate, but the wrapper phrase itself pollutes every occurrence regardless of dedup. |
| 2 | `[QUOTE=user;postid]...[/QUOTE]` BBCode quote tags | included in #3's 1.1%, but semantically the *same problem* as #1 | Strip tag markup, same as #1's treatment | A second, older/newer quoting mechanism doing the same thing as #1. Confirmed **nesting**: one example had a `[QUOTE=...]` wrapping an inner `اقتباس:` wrapper (someone quoting a quote) — strip both patterns, then re-check (loop until no more matches), don't assume one pass is enough. |
| 3 | Other BBCode tags (`[COLOR]`, `[SIZE]`, `[FONT]`, `[IMG]`, `[/B]`, `[table]`, ...) | 1.1% | Strip tag markup, keep inner text | Leftover markup that didn't get rendered to plain text — no semantic content in the tags themselves. |
| 4 | URL | 1.6% | Replace with `[URL]` placeholder | Same anonymization treatment as YouTube — more common here (forum posts share links more than YouTube comments do). |
| 5 | HTML entities (`&amp;`, `&nbsp;`, ...) | **0.0%** | No rule needed | Checked directly — none found in this sample. Not worth the complexity unless a larger sample later shows otherwise. |
| 6 | 3+ consecutive emoji | 0.8% | Collapse to max 2 (same rule as YouTube) | Much rarer here than YouTube (0.8% vs 17.3%, mean 0.11 vs 1.57 emoji/doc) — forum culture uses far less emoji. Low priority for this source specifically, but cheap to share the same rule/code. |
| 7a | Tatweel/kashida decoration (`ـ`, U+0640) repeated | included in #7b's 16.8%, but a distinct case | **Strip entirely** (not collapse) | New finding, not seen in the YouTube sample: the single most common "elongated run" here is the Arabic *tatweel* character (`ـــ`, `ـــــ`, ...) — purely decorative text-stretching with **zero semantic content**, unlike a genuinely repeated letter. Different treatment from 7b: delete, don't collapse-and-keep. |
| 7b | Elongated real letters (laughter `هههه`, emphasis `اااا`) | 16.8% total (tatweel included) | Collapse to max 2–3 repeats, not delete | Same "preserve expressiveness" policy as YouTube. **Caveat found here**: the raw regex also flags coincidental repeats inside URLs (`www`) and numbers (`000` in `1,500,000`) as false positives — exclude matches inside URL spans and digit-only runs before applying the collapse. |
| 8 | Excessive punctuation run (same char 3+×) | 24.0% | **Split by character**: periods → cap at exactly 3 (`...`, standard ellipsis); `!`/`؟`/`,` → collapse to max 2 | Different nuance from YouTube: most of this 24% is legitimate Arabic prose using `...` as a rhetorical pause (very common, e.g. `"اللهم لا تخرجنا من يومنا هذا إلا... اللهم لا تخرجنا... "`), not spam emphasis. Treating `.` like `!`/`؟` would over-normalize normal writing style — periods need their own, gentler rule. |
| 9 | Forum "who's online" widget leakage (`أكبر تواجد بالمنتدى كان...` + username list) | 0.4% | Strip entire block via a dedicated pattern match on the phrase | Small but confirmed recurring (not a one-off) — 100% content-free auto-generated stat block, not real post text. **Also worth a scraper-side check**: this looks like sidebar/footer widget content that leaked into post extraction (`src/darija_forum/parse.py`) rather than something that belongs in `list_posts()`'s output at all — cleaning can patch it, but the cleaner fix might be at the scraping boundary. |
| 10 | Near-empty after stripping emoji/punctuation | **0.0%** | Keep the min-length filter (shared with YouTube) but expect it to rarely fire here | djelfa posts are far longer on average than YouTube comments (median residual 127 chars vs. YouTube's ~25) — this filter matters much less for this source in practice, even though the rule itself should stay shared/generic. |

**Order matters** (updated for this source): strip forum-widget leakage (#9) and quote wrappers (#1, #2 — loop until stable) *before* anything else, since they're large blocks that would otherwise skew length-based decisions (e.g. #10) and dilute other pattern-prevalence checks. Strip tatweel (#7a) before the general elongation collapse (#7b) so it doesn't get miscounted as a "real" elongated letter.

## 12. Apply `clean_text.py` and validate

In [11]:
import sys

sys.path.insert(0, str(ROOT / "Mountada_djelfa_scrap" / "src"))
from darija_forum import clean_text  # noqa: E402

df["cleaned"] = df["text"].apply(clean_text.clean)
dropped = df["cleaned"].isna()

print(f"dropped as near-empty: {dropped.sum()}/{len(df)} ({dropped.mean() * 100:.1f}%)")

dropped as near-empty: 3/1000 (0.3%)

In [12]:
print("--- before/after (random sample) ---")
for _, row in df[~dropped].sample(12, random_state=1).iterrows():
    print(f"BEFORE: {row['text'][:200]!r}")
    print(f"AFTER:  {row['cleaned'][:200]!r}")
    print()

--- before/after (random sample) ---
BEFORE: 'اقتباس:\nالمشاركة الأصلية كتبت بواسطة soha1987\nردي بالك حنا الشاوية نعايرو بزااااااف هههههههه راني نتمسخر برك\nمالا مافهمتيش واش قوتلك معناها سلمي على ماماك\nربي يخليك لماماك مريومة وربي يشفيك ويعطيك مات'
AFTER:  'ردي بالك حنا الشاوية نعايرو بزااف هه راني نتمسخر برك\nمالا مافهمتيش واش قوتلك معناها سلمي على ماماك\nربي يخليك لماماك مريومة وربي يشفيك ويعطيك ماتتمناي\nسلام\nالسلام عليكم\nالصراحة غير تولو تهدرو باللهجة ن'

BEFORE: 'تاج الوقار\nالسلام عليكم ورحمة الله وبركاته.\nمرحبا بكِ في هذا المتصفح " ضيفة تحت المجهر"\nوحتى لا آخذ من وقتكِ الشيء الكثير.\nيا فاضلة،\nيقول أحد أساطين التربية وأصول التدريس:\nLes meilleurs enseignants so'
AFTER:  'تاج الوقار\nالسلام عليكم ورحمة الله وبركاته.\nمرحبا بك في هذا المتصفح " ضيفة تحت المجهر"\nوحتى لا آخذ من وقتك الشيء الكثير.\nيا فاضلة،\nيقول أحد أساطين التربية وأصول التدريس:\nLes meilleurs enseignants sont'

BEFORE: 'السلام عليكم و رحمة اله تعالى و بركاته\nاهلا و سهلا بالاخت شاهندا بيننا\nلا اسئلة لدي ا

### 12a. Confirm each targeted pattern's prevalence actually dropped

In [13]:
cleaned_series = df.loc[~dropped, "cleaned"]

def prevalence(pattern, series):
    hits = series.apply(lambda t: bool(pattern.search(t))).sum()
    return hits, hits / len(series) * 100

LETTER_ELONGATION_RE = re.compile(r"([^\W\d_])\1{2,}", re.UNICODE)

# NOTE: quote-wrapper/BBCode/tatweel/emoji-run checks below reuse clean_text's
# own patterns rather than this notebook's earlier (cruder/non-grapheme-aware)
# ones — same calibration lesson learned on the YouTube side (see that
# notebook's Section 10): the *implementation's* pattern is the correct
# yardstick for "did the implementation's own rule actually fire".
checks = {
    "Quote wrapper (should be ~0)": clean_text.QUOTE_WRAPPER_RE,
    "BBCode tags (should be ~0)": clean_text.BBCODE_RE,
    "Tatweel (should be ~0)": clean_text.TATWEEL_RE,
    "URLs (should be ~0)": URL_RE,
    "Forum widget leakage (should be ~0)": clean_text.WIDGET_LEAK_RE,
    "3+ consecutive emoji, grapheme-aware (should be ~0 — collapsed to 2)": clean_text.EMOJI_RUN_RE,
    "3+ elongated letters, digits excluded (should be ~0 — collapsed to 2)": LETTER_ELONGATION_RE,
    "3+ repeated !/؟/,/، (should be ~0 — collapsed to 2)": re.compile(r"([!?؟,،])\1{2,}"),
}
for name, pattern in checks.items():
    hits, pct = prevalence(pattern, cleaned_series)
    print(f"{name}: {hits}/{len(cleaned_series)} ({pct:.1f}%)")

period_runs = [m.group(0) for t in cleaned_series for m in re.finditer(r"\.{2,}", t)]
print(f"\nremaining period runs after cleaning: {Counter(period_runs)}")

Quote wrapper (should be ~0): 0/997 (0.0%)
BBCode tags (should be ~0): 0/997 (0.0%)
Tatweel (should be ~0): 0/997 (0.0%)
URLs (should be ~0): 0/997 (0.0%)
Forum widget leakage (should be ~0): 0/997 (0.0%)
3+ consecutive emoji, grapheme-aware (should be ~0 — collapsed to 2): 0/997 (0.0%)
3+ elongated letters, digits excluded (should be ~0 — collapsed to 2): 0/997 (0.0%)
3+ repeated !/؟/,/، (should be ~0 — collapsed to 2): 0/997 (0.0%)

remaining period runs after cleaning: Counter({'...': 1133})


### 12b. Sanity-check the dropped documents

In [14]:
for text in df.loc[dropped, "text"].sample(min(20, dropped.sum()), random_state=2):
    print(f"- {text[:150]!r}")

- 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 11:15\n\u200fزهرة المسيلة, \u200fالأخضر48, \u200fMOHAMMED.AMIN21, \u200fنسيم الشوق, \u200fmisa39, \u200fbahi65b+, \u200fnadir2006'
- 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 12:15\n\u200fزهرة المسيلة, \u200fabouyounes, \u200ftorab12, \u200f*عبدالرحمن*, \u200fkhalide, \u200fnono3, \u200fasmt, \u200fabdou-lad'
- 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 11:15\n\u200fزهرة المسيلة, \u200fhattou ahmed khalil, \u200fshouaib, \u200fجلاليتو, \u200fhami_luis, \u200fel matadoor, \u200fche'


### 12c. Spot-check the djelfa-specific transformations directly

The general random sample above is unlikely to hit the rarer patterns (widget leakage 0.4%, BBCode 1.1%) — targeting docs that originally matched each pattern and inspecting before/after directly.

In [15]:
targeted = {
    "widget leakage": WIDGET_LEAK_RE,
    "BBCode": BBCODE_RE,
    "tatweel": re.compile("ـ+"),
}
for name, pattern in targeted.items():
    orig_mask = df["text"].apply(lambda t: bool(pattern.search(t)))
    print(f"=== {name}: {orig_mask.sum()} originally-matching docs ===")
    for _, row in df[orig_mask].head(3).iterrows():
        after = row["cleaned"] if pd.notna(row["cleaned"]) else "<DROPPED as near-empty>"
        print(f"BEFORE: {row['text'][:200]!r}")
        print(f"AFTER:  {after[:200]!r}")
        print()

=== widget leakage: 4 originally-matching docs ===
BEFORE: 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 11:15\n\u200fزهرة المسيلة, \u200fhattou ahmed khalil, \u200fshouaib, \u200fجلاليتو, \u200fhami_luis, \u200fel matadoor, \u200fchekired, \u200fنائلة, \u200fبوقرة محمد, \u200fahmedabdikarim, \u200fYAZI'
AFTER:  '<DROPPED as near-empty>'

BEFORE: 'مشاهدة المتواجدون الآن\nأكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 11:15\n\u200fزهرة المسيلة, \u200fdz-yac, \u200fأبو سارة عبد اللطيف, \u200fsaid wail, \u200fأم إسلام, \u200fAMIRA76, \u200fأستاذ شاوي, \u200fArkham, \u200fH@liM, \u200fأبوع'
AFTER:  'مشاهدة المتواجدون الآن'

BEFORE: 'أكبر تواجد بالمنتدى كان: 25,091 بتاريخ 2019-04-29 الساعة 12:15\n\u200fزهرة المسيلة, \u200fabouyounes, \u200ftorab12, \u200f*عبدالرحمن*, \u200fkhalide, \u200fnono3, \u200fasmt, \u200fabdou-lad, \u200fأبو أشْرف, \u200flaouiyacine, \u200fbissa40, \u200fchanfawa, \u200f'
AFTER:  '<DROPPED as near-empty>'

=== BBCode: 11 originally-matching doc

## 13. Amendments found during validation

Issues found by actually running `clean_text.py` against the sample and reading the output — not caught by the original rule-derivation pass (Sections 1–11):

| Amendment | What was found | Fix |
|---|---|---|
| **`[URL]` placeholder collides with `BBCODE_RE` (real bug, latent idempotency hazard)** | `replace_urls` inserts the literal token `[URL]` as an anonymization placeholder. That token is itself shaped exactly like a BBCode tag (`\[[a-zA-Z]+\]`), so re-running `BBCODE_RE` against already-cleaned text — which is exactly what the Section 12a validation check does — flagged 16/997 (1.6%) "leftover BBCode" hits that were all `[URL]` tokens, not real markup. Within a single `clean()` call this isn't actively destructive (`strip_bbcode` runs before `replace_urls`, so nothing is double-processed), but it's a landmine: if `clean()` is ever re-applied to already-cleaned text (e.g. accidentally re-run in a later pipeline step), `strip_bbcode` would eat every `[URL]` placeholder it had itself produced. | Added a negative lookahead to `BBCODE_RE` — `\[/?(?!URL\])[a-zA-Z]+...` — so it never matches the anonymization placeholder's own shape, while still matching real forum tags including lowercase `[url]...[/url]` and `[URL=...]` (verified directly: both still stripped correctly). |
| Malformed BBCode tags with an embedded newline (minor, low-prevalence) | 2/1000 sampled docs had a BBCode closing tag mangled by a stray newline from the source (`"[/QUOTE\n]"`, `"[/cent\ner]"` — the latter with the newline splitting the tag *name* itself, "center" → "cent"+newline+"er"). The original `BBCODE_RE` required the tag name to be immediately followed by `]`, so these survived stripping. | Added `\s*` before the closing bracket, which fixes the `"[/QUOTE\n]"` case (trailing whitespace before `]`) at effectively zero risk of new false matches. The `"[/cent\ner]"` case — where the newline splits the tag name itself — is left unfixed: catching it would require allowing whitespace *inside* `[a-zA-Z]+`, which risks matching unrelated bracketed text spanning unrelated lines. At 1/1000 prevalence this is a rare scraper/source-formatting artifact, not worth that regex risk; noting it here rather than in `clean_text.py` as a known, accepted gap. |

**Result after fixes** (Section 12a re-run): every targeted pattern — quote wrapper, BBCode, tatweel, URLs, forum-widget leakage, 3+ emoji runs, 3+ elongation, 3+ other-punctuation — is at 0.0% on the cleaned sample. Only 0.3% of docs (3/1000) were dropped as near-empty, consistent with djelfa posts being much longer on average than YouTube comments (Section 10's finding). The remaining `...` matches (1,133 of them) are exactly 3-dot ellipses, as intended — not leftover excessive punctuation.

**No dedicated rule needed beyond what Section 11 already proposed** — unlike the YouTube notebook, no *new* character-class rule (like the compound-emoji grapheme fix there) was needed here; both amendments above are about the BBCode rule's own pattern, not a new noise category.

## 14. Arabic diacritics (tachkil) — added after initial validation

Not part of the original rule-derivation pass (Sections 1–11); added later
after discussing whether tachkil (tashkeel: fatha, damma, kasra, sukun,
shadda, tanwin, etc.) and newlines were worth touching. Conclusion:
newlines carry real structure in this forum's longer, list-formatted posts
and stay untouched, but tachkil isn't a Darija feature — casual Darija is
written essentially undiacritized — so diacritized fragments are almost
always religious/poetic quotes, formal text, or usernames bleeding in via
the quote wrapper, not deliberate dialectal expression, and just fragment
tokenization (`"السلام"` vs `"السَّلَام"` as distinct tokens for the same
word). Much more prevalent here than on YouTube (26.9% of this sample vs.
0.6% there), consistent with djelfa's heavier religious/formal-register
content.

In [16]:
# Reuses clean_text.TACHKIL_RE directly (imported in Section 12) rather than
# retyping the Unicode range here -- typing raw diacritic characters mixed
# with regex syntax is a real risk (confirmed separately: a first attempt
# at this same range got silently scrambled by RTL-aware text handling into
# a much wider, wrong range that would have deleted Arabic-Indic digits).
report("Arabic diacritics (tachkil)", clean_text.TACHKIL_RE)

=== Arabic diacritics (tachkil): 269/1000 (26.9%) ===
- 'السؤال: أقرأ في بعض المنتديات أن جميع علامات الساعة الصغرى قد ظهرت ويكتبون بأنه لم يبق شيء عن قيام الساعة فما مدى صحة كلامهم ؟ . الجواب : الحمد لله يقسِّم بعض العلماء علامات الساعات إلى كبرى ، وصغرى ،'
- '"الحمد لله على قضائه، وعلى ستره، وعلى رحمته، وعلى كلِّ نعمة ننعم بها.. الحمد لله حمد الشاكرين، والشكر لله شكر الحامدين، والحمد لله على لُطفه، وعلى كرمه، وعلى كلِّ خيره.. اللَّهُمَّ لك الحمد على عطاءك '
- 'اقتباس: المشاركة الأصلية كتبت بواسطة * أبو فراس * وليد ضيف تحت المجهر -------- أهلا بالأخ  وليد (الغامض) صحافي المنتدى المتألق ضيفا محترما مرحبا به في صفحة الاعتراف صراحة تعجبني جدا مواضيعك وأكثر ما ي'
- 'لرفع مستوى السعادة في حياتك، تعلّم ثلاث هوايات : هواية تجلب لك المال ، و هواية تجلب لك الصحة ، و هواية تجلب لك الراحة ... 👌 💚💙❤️'
- '- عَنْ عَائِشَةَ ، أَنّ رَسُولَ اللَّهِ صَلَّى اللَّهُ عَلَيْهِ وَسَلَّمَ قَالَ : " مَا مِنْ مُسْلِمٍ مِنَ الْمُسْلِمِينَ يَمُوتُ يُصَلِّي عَلَيْهِ أُمَّةٌ مِنَ النَّاسِ يَبْلُغُونَ أَنْ يَكُونُو

0      False
1       True
2       True
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

In [17]:
# Before/after on docs that originally had tachkil, using the already-
# computed `cleaned` column from Section 12 (re-running this notebook
# top-to-bottom re-imports clean_text.py fresh, so `cleaned` reflects the
# current implementation including tachkil-stripping).
had_tachkil = df["text"].apply(lambda t: bool(clean_text.TACHKIL_RE.search(t)))
print(f"docs with tachkil: {had_tachkil.sum()}/{len(df)}\n")
for _, row in df[had_tachkil & ~dropped].head(6).iterrows():
    print(f"BEFORE: {row['text'][:200]!r}")
    print(f"AFTER:  {row['cleaned'][:200]!r}")
    print()

# Confirm no tachkil codepoints survive in cleaned output, and that ASCII
# and Arabic-Indic digits are untouched (the regex must not overreach into
# the digit range -- confirmed directly, not just assumed).
remaining = cleaned_series.apply(lambda t: bool(clean_text.TACHKIL_RE.search(t))).sum()
print(f"remaining tachkil after cleaning: {remaining}/{len(cleaned_series)}")
print("digit preservation check:", clean_text.clean("عندي 1500 دج و ١٥٠٠ دج"))

docs with tachkil: 269/1000

BEFORE: 'اقتباس:\nالمشاركة الأصلية كتبت بواسطة حَمزة\nو عليكم السّلام\nمرحبا منال ، اذا تريدين معرفة تفاصيل القصة المخيفة ،، حسنا و لكن عليك أن تكوني قوية لانها مرعبة ...\nهناك قصّتين الاولى تجاوزتها ، في بداية تع'
AFTER:  'و عليكم السلام\nمرحبا منال ، اذا تريدين معرفة تفاصيل القصة المخيفة ،، حسنا و لكن عليك أن تكوني قوية لانها مرعبة ...\nهناك قصتين الاولى تجاوزتها ، في بداية تعلمي القيادة طلبت من أبي ان يعلمني كدعم كيف لا'

BEFORE: 'اقتباس:\nالمشاركة الأصلية كتبت بواسطة صَمْـتْــــ~\nالسّلام عليكم ورحمة الله وبركاته\nأهلا بكِ أختنا\nتسنيم\nفي رِحاب هذا الرُّكن الأخويّ،\nالذي أُضيءَ بطيبِ حضورِك\nوحتى لا أثقل عليكِ فسأكتفي بسؤالٍ واحدٍ .'
AFTER:  'السلام عليكم ورحمة الله وبركاته\nأهلا بك أختنا\nتسنيم\nفي رحاب هذا الركن الأخوي،\nالذي أضيء بطيب حضورك\nوحتى لا أثقل عليك فسأكتفي بسؤال واحد ...\nمع احترامي المسبق لردك\nو عليكم السلام و رحمة الله و بركاته\nأ'

BEFORE: 'مساء القلوب النيّرة ... الطيّبة ...\nوالمليئة بالحب و اللطف ...\n💚💙❤️'
AFTER:  'مساء القلوب الن